# Scraper Alcampo

Este notebook sigue la misma lógica que los scrapers de **Mercadona** y **DIA**:

1. Parte siempre de la página principal de Alcampo.
2. Localiza el buscador de la propia web.
3. Escribe cada producto en el buscador.
4. Espera a que carguen los resultados.
5. Hace scroll para cargar todos los productos disponibles.
6. Extrae nombre, precio, precio por unidad, URL e ID del producto.
7. Une todas las búsquedas en un único dataset.
8. Elimina duplicados por `product_id`.
9. Normaliza las unidades.
10. Guarda:
   - un único CSV raw en `data/raw`
   - un único CSV procesado en `data/processed`

**Primero se ejecuta en modo prueba con `leche`.**  
Cuando funcione correctamente, cambia `TEST_MODE = False` para lanzar las 20 búsquedas.

In [38]:
import time
import re
from pathlib import Path
from datetime import datetime
from urllib.parse import urljoin

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException


HOME_URL = "https://www.compraonline.alcampo.es/"

SEARCH_TERMS = [
    "leche",
    "huevos",
    "manzana",
    "plátano",
    "tomate",
    "arroz",
    "pasta",
    "pan",
    "pollo",
    "carne picada",
    "salmón",
    "yogur",
    "queso",
    "agua",
    "zumo",
    "atún",
    "garbanzos",
    "aceite",
    "detergente",
    "papel higiénico",
]

# --------------------------------------------------
# MODO DE PRUEBA
# --------------------------------------------------

TEST_MODE = True
TEST_TERMS = ["leche"]

TERMS_TO_SCRAPE = TEST_TERMS if TEST_MODE else SEARCH_TERMS

MAX_SCROLLS = 40
SCROLL_PAUSE = 1.5
STABLE_ROUNDS_TO_STOP = 3

print("Búsquedas que se ejecutarán:", TERMS_TO_SCRAPE)

Búsquedas que se ejecutarán: ['leche']


In [39]:
# --------------------------------------------------
# LOCALIZAR CARPETAS DEL PROYECTO
# --------------------------------------------------

def find_project_root():
    current = Path.cwd().resolve()

    for path in (current, *current.parents):
        if path.name == "supermarket_project":
            return path

    # Si el proyecto tiene otro nombre pero el notebook
    # se ejecuta desde /scrapers, usamos su carpeta padre.
    if current.name == "scrapers":
        return current.parent

    raise FileNotFoundError(
        "No se ha podido localizar la raíz del proyecto. "
        "Ejecuta el notebook desde la carpeta scrapers."
    )


BASE_DIR = find_project_root()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Proyecto:", BASE_DIR)
print("Raw:", RAW_DIR)
print("Processed:", PROCESSED_DIR)

Proyecto: C:\Users\paula\OneDrive\Documentos\uc3m\Cuarto Año\Analitica Web\Supermarket_Project
Raw: C:\Users\paula\OneDrive\Documentos\uc3m\Cuarto Año\Analitica Web\Supermarket_Project\data\raw
Processed: C:\Users\paula\OneDrive\Documentos\uc3m\Cuarto Año\Analitica Web\Supermarket_Project\data\processed


In [47]:
# --------------------------------------------------
# DRIVER Y FUNCIONES DE NAVEGACIÓN
# --------------------------------------------------

from selenium.webdriver.chrome.options import Options
from selenium import webdriver

def start_alcampo_driver():
    options = Options()
    options.add_argument("--window-size=1920,1080")
    
    # --- ARGUMENTOS ANTI-DETECCIÓN ---
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(options=options)
    
    # Ocultar la propiedad navigator.webdriver mediante JavaScript
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    driver.get(HOME_URL)
    return driver


def accept_alcampo_cookies(driver):
    xpaths = [
        "//button[contains(., 'Aceptar todas')]",
        "//button[contains(., 'Aceptar todo')]",
        "//button[contains(., 'Aceptar')]",
        "//button[contains(., 'Permitir todas')]",
    ]

    for xpath in xpaths:
        buttons = driver.find_elements(By.XPATH, xpath)

        for button in buttons:
            if button.is_displayed() and button.is_enabled():
                try:
                    button.click()
                    print("Cookies aceptadas.")
                    return
                except Exception:
                    pass

    print("No hay banner de cookies visible.")


def assert_no_security_challenge(driver):
    body_text = driver.find_element(By.TAG_NAME, "body").text.lower()

    messages = [
        "captcha",
        "access denied",
        "confirma que no eres un robot",
        "comprueba que eres humano",
        "verify you are human",
    ]

    if any(message in body_text for message in messages):
        raise RuntimeError(
            "Se ha detectado una posible verificación de seguridad."
        )


def find_search_box(driver):
    # Intentamos primero atributos semánticos.
    selectors = [
        'input[type="search"]',
        'input[placeholder*="Buscar en Alcampo"]',
        'input[placeholder*="Buscar"]',
        'input[aria-label*="Buscar"]',
        'input[name*="search"]',
        'input[id*="search"]',
    ]

    def visible_search_box():
        for selector in selectors:
            elements = driver.find_elements(By.CSS_SELECTOR, selector)

            for element in elements:
                if element.is_displayed() and element.is_enabled():
                    return element

        return False

    try:
        return WebDriverWait(driver, 12).until(
            lambda _: visible_search_box()
        )

    except TimeoutException:
        diagnostics = []

        for element in driver.find_elements(By.TAG_NAME, "input"):
            if element.is_displayed():
                diagnostics.append({
                    "type": element.get_attribute("type"),
                    "name": element.get_attribute("name"),
                    "id": element.get_attribute("id"),
                    "placeholder": element.get_attribute("placeholder"),
                    "aria-label": element.get_attribute("aria-label"),
                })

        raise RuntimeError(
            "No se ha encontrado el buscador de Alcampo. "
            f"Inputs visibles: {diagnostics}"
        )

In [50]:
# --------------------------------------------------
# BÚSQUEDA A TRAVÉS DE LA INTERFAZ DE ALCAMPO
# --------------------------------------------------

def count_product_links(driver):
    links = driver.find_elements(
        By.CSS_SELECTOR,
        'a[href*="/products/"]'
    )

    hrefs = {
        link.get_attribute("href")
        for link in links
        if link.get_attribute("href")
    }

    return len(hrefs)


from urllib.parse import quote

from urllib.parse import quote

def search_alcampo(driver, term):
    assert_no_security_challenge(driver)
    
    encoded_term = quote(term)
    # Probemos con el parámetro habitual de búsqueda de su web
    search_url = f"https://www.compraonline.alcampo.es/search?q={encoded_term}"
    
    print(f"Navegando directamente a la URL de búsqueda: {search_url}")
    driver.get(search_url)
    
    time.sleep(4)  # Damos más margen para que cargue la SPA
    assert_no_security_challenge(driver)
    
    # Imprimimos información de depuración para ver qué ha cargado
    print("Título de la página actual:", driver.title)
    print("Número de enlaces /products/ encontrados:", count_product_links(driver))

    def search_loaded(d):
        assert_no_security_challenge(d)
        
        # Damos un pequeño respiro y comprobamos si ya hay productos cargados
        new_count = count_product_links(d)
        if new_count > 0:
            return True
            
        return False

    try:
        WebDriverWait(
            driver,
            20,
            poll_frequency=0.4
        ).until(search_loaded)

    except TimeoutException:
        # Puede ocurrir que una búsqueda devuelva 0 resultados.
        body_text = driver.find_element(By.TAG_NAME, "body").text.lower()

        if (
            "no hemos encontrado" in body_text
            or "sin resultados" in body_text
            or "0 resultados" in body_text
        ):
            print(f"Sin resultados para: {term}")
            return

        raise TimeoutException(
            f"No se pudo confirmar la carga de la búsqueda '{term}'."
        )

    print(
        f"Buscado mediante la interfaz: {term}\n"
        f"URL actual: {driver.current_url}"
    )

In [51]:
# --------------------------------------------------
# CARGAR TODOS LOS RESULTADOS HACIENDO SCROLL
# --------------------------------------------------

def load_all_alcampo_products(
    driver,
    max_scrolls=MAX_SCROLLS,
    stable_rounds_to_stop=STABLE_ROUNDS_TO_STOP,
):
    previous_count = count_product_links(driver)
    stable_rounds = 0

    print("Productos iniciales detectados:", previous_count)

    for scroll_number in range(1, max_scrolls + 1):
        driver.execute_script(
            "window.scrollTo(0, document.body.scrollHeight);"
        )

        time.sleep(SCROLL_PAUSE)
        assert_no_security_challenge(driver)

        current_count = count_product_links(driver)

        print(
            f"Scroll {scroll_number}: "
            f"{previous_count} → {current_count}"
        )

        if current_count > previous_count:
            previous_count = current_count
            stable_rounds = 0

        else:
            stable_rounds += 1

        if stable_rounds >= stable_rounds_to_stop:
            print(
                "No aparecen nuevos productos. "
                "Se considera finalizada la carga."
            )

            return current_count, "stable"

    return previous_count, "max_scrolls_reached"

In [52]:
# --------------------------------------------------
# FUNCIONES DE LIMPIEZA
# --------------------------------------------------

def clean_number(text):
    if text is None:
        return None

    match = re.search(
        r"\d+(?:[.,]\d+)?",
        str(text).replace("\xa0", " ")
    )

    if not match:
        return None

    return float(
        match.group()
        .replace(",", ".")
    )


def normalize_unit_name(unit_text):
    if not unit_text:
        return None

    unit = (
        unit_text
        .strip()
        .lower()
        .replace(".", "")
    )

    mappings = {
        "kg": "KILO",
        "kilo": "KILO",
        "kilogramo": "KILO",
        "kilogramos": "KILO",

        "l": "LITRO",
        "litro": "LITRO",
        "litros": "LITRO",

        "unidad": "UNIDAD",
        "ud": "UNIDAD",
        "uds": "UNIDAD",

        "docena": "DOCENA",
        "lavado": "LAVADO",
        "lavados": "LAVADO",
    }

    return mappings.get(unit, unit.upper())


def extract_price_per_unit(text):
    if not text:
        return None, None, None

    cleaned = (
        str(text)
        .replace("\xa0", " ")
        .strip()
    )

    # Formato típico:
    # 1000ml(0,83 € por litro)
    match = re.search(
        r"(\d+(?:[.,]\d+)?)\s*€\s+por\s+"
        r"(litro|litros|kg|kilo|kilogramo|kilogramos|"
        r"unidad|ud|uds|docena|lavado|lavados)",
        cleaned,
        flags=re.IGNORECASE,
    )

    if match:
        value = float(
            match.group(1)
            .replace(",", ".")
        )

        unit = normalize_unit_name(
            match.group(2)
        )

        return value, unit, match.group(0)

    # Posibles formatos por 100 ml / 100 g
    match_100 = re.search(
        r"(\d+(?:[.,]\d+)?)\s*€\s*(?:por|/)\s*100\s*"
        r"(ml|g|gr)",
        cleaned,
        flags=re.IGNORECASE,
    )

    if match_100:
        value = float(
            match_100.group(1)
            .replace(",", ".")
        )

        measure = match_100.group(2).lower()

        if measure == "ml":
            return value * 10, "LITRO", match_100.group(0)

        return value * 10, "KILO", match_100.group(0)

    return None, None, None


def extract_product_id(url):
    if not url:
        return None

    match = re.search(
        r"/products/[^/]+/(\d+)",
        url
    )

    return match.group(1) if match else None

In [53]:
# --------------------------------------------------
# EXTRAER PRODUCTOS DE LA PÁGINA ACTUAL
# --------------------------------------------------

def find_product_container(anchor):
    current = anchor

    for _ in range(8):
        current = current.parent

        if current is None:
            break

        text = current.get_text(" ", strip=True)

        # Una tarjeta de producto suele contener precio
        # y el botón "Añadir".
        if (
            "Precio" in text
            and "Añadir" in text
            and len(text) < 2500
        ):
            return current

    return anchor.parent


def extract_alcampo_products(driver, searched_item):
    soup = BeautifulSoup(
        driver.page_source,
        "html.parser"
    )

    anchors = soup.select(
        'a[href*="/products/"]'
    )

    records = []
    seen_urls = set()
    scraped_at = datetime.now().isoformat(
        timespec="seconds"
    )

    for anchor in anchors:
        href = anchor.get("href")

        if not href:
            continue

        url = urljoin(HOME_URL, href)

        # El mismo producto puede tener más de un enlace
        # dentro de la tarjeta.
        if url in seen_urls:
            continue

        seen_urls.add(url)

        container = find_product_container(anchor)

        if container is None:
            continue

        text = container.get_text(
            " ",
            strip=True
        )

        # --------------------------------------------------
        # NOMBRE
        # --------------------------------------------------

        name = (
            anchor.get("aria-label")
            or anchor.get("title")
            or anchor.get_text(" ", strip=True)
        )

        # Si el enlace es una imagen o viene vacío,
        # buscamos un texto útil dentro de la tarjeta.
        if not name:
            candidate = container.find(
                ["h2", "h3", "h4"]
            )

            if candidate:
                name = candidate.get_text(
                    " ",
                    strip=True
                )

        # --------------------------------------------------
        # PRECIO
        # --------------------------------------------------

        price = None
        price_raw = None

        price_match = re.search(
            r"Precio\s+(\d+(?:[.,]\d+)?)\s*€",
            text,
            flags=re.IGNORECASE,
        )

        if price_match:
            price_raw = price_match.group(0)
            price = float(
                price_match.group(1)
                .replace(",", ".")
            )

        # --------------------------------------------------
        # PRECIO POR UNIDAD
        # --------------------------------------------------

        price_per_unit, unit, ppu_raw = (
            extract_price_per_unit(text)
        )

        # --------------------------------------------------
        # TAMAÑO / CANTIDAD MOSTRADA
        # --------------------------------------------------

        quantity_raw = None

        quantity_match = re.search(
            r"\b\d+(?:[.,]\d+)?\s*"
            r"(?:ml|cl|l|g|gr|kg|uds?|unidades?)\b",
            text,
            flags=re.IGNORECASE,
        )

        if quantity_match:
            quantity_raw = quantity_match.group(0)

        product_id = extract_product_id(url)

        # No guardamos enlaces que claramente no parecen
        # productos utilizables.
        if not name or price is None:
            continue

        records.append({
            "searched_item": searched_item,
            "supermarket": "Alcampo",
            "product_id": product_id,
            "name": name,
            "quantity_raw": quantity_raw,
            "price_raw": price_raw,
            "price": price,
            "price_per_unit_raw": ppu_raw,
            "price_per_unit": price_per_unit,
            "unit": unit,
            "url": url,
            "scraped_at": scraped_at,
        })

    df = pd.DataFrame(records)

    print(
        f"Enlaces /products/ detectados: {len(anchors)}"
    )
    print(
        f"Productos extraídos: {len(df)}"
    )

    return df

In [54]:
# --------------------------------------------------
# PRUEBA CON UNA ÚNICA BÚSQUEDA
# --------------------------------------------------

driver = start_alcampo_driver()

try:
    time.sleep(3)

    accept_alcampo_cookies(driver)

    term = TERMS_TO_SCRAPE[0]

    search_alcampo(
        driver,
        term
    )

    loaded_count, load_status = (
        load_all_alcampo_products(driver)
    )

    df_test = extract_alcampo_products(
        driver,
        term
    )

    print("\nEstado de carga:", load_status)
    print("Productos cargados:", loaded_count)
    print("Filas extraídas:", len(df_test))

    display(df_test.head(20))

finally:
    driver.quit()

Cookies aceptadas.
Navegando directamente a la URL de búsqueda: https://www.compraonline.alcampo.es/search?q=leche
Título de la página actual: Resultados de la búsqueda para leche - Alcampo supermercado online
Número de enlaces /products/ encontrados: 10
Buscado mediante la interfaz: leche
URL actual: https://www.compraonline.alcampo.es/search?q=leche
Productos iniciales detectados: 10
Scroll 1: 10 → 16
Scroll 2: 16 → 16
Scroll 3: 16 → 16
Scroll 4: 16 → 16
No aparecen nuevos productos. Se considera finalizada la carga.
Enlaces /products/ detectados: 32
Productos extraídos: 16

Estado de carga: stable
Productos cargados: 16
Filas extraídas: 16


,searched_item,supermarket,product_id,name,quantity_raw,price_raw,price,price_per_unit_raw,price_per_unit,unit,url,scraped_at
0,leche,Alcampo,54180,AUCHAN Leche semidesnatada de vaca 6 x 1l Prod...,1l,"Precio 5,28 €",5.28,"0,88 € por litro",0.88,LITRO,https://www.compraonline.alcampo.es/products/a...,2026-10-02T13:21:15
1,leche,Alcampo,50857,CENTRAL LECHERA ASTURIANA Leche semidesnatada ...,1 l,"Precio 1,04 €",1.04,"1,04 € por litro",1.04,LITRO,https://www.compraonline.alcampo.es/products/c...,2026-10-02T13:21:15
2,leche,Alcampo,566225,CENTRAL LECHERA ASTURIANA Leche semidesnatada ...,200 ml,"Precio 1,21 €",1.21,"2,02 € por litro",2.02,LITRO,https://www.compraonline.alcampo.es/products/c...,2026-10-02T13:21:15
3,leche,Alcampo,89826,CENTRAL LECHERA ASTURIANA Leche de vaca semide...,500 ml,"Precio 4,44 €",4.44,"1,48 € por litro",1.48,LITRO,https://www.compraonline.alcampo.es/products/c...,2026-10-02T13:21:15
4,leche,Alcampo,54505,PASCUAL Leche semidesnatada procedente de anim...,1l,"Precio 6,78 €",6.78,"1,13 € por litro",1.13,LITRO,https://www.compraonline.alcampo.es/products/p...,2026-10-02T13:21:15
5,leche,Alcampo,51119,CENTRAL LECHERA ASTURIANA Leche semidesnatada ...,2.2 l,"Precio 2,18 €",2.18,"0,99 € por litro",0.99,LITRO,https://www.compraonline.alcampo.es/products/c...,2026-10-02T13:21:15
6,leche,Alcampo,535982,PULEVA Eco Leche semidesnatada ecológica de v...,1 l,"Precio 1,70 €",1.70,"1,70 € por litro",1.70,LITRO,https://www.compraonline.alcampo.es/products/p...,2026-10-02T13:21:15
7,leche,Alcampo,51327,TIERRA DE SABOR Leche semidesnatada de vaca 6 ...,1 l,"Precio 6,72 €",6.72,"1,12 € por litro",1.12,LITRO,https://www.compraonline.alcampo.es/products/t...,2026-10-02T13:21:15
8,leche,Alcampo,53049,PULEVA Leche semidesnatada de vacas con alimen...,1 l,"Precio 6,00 €",6.00,"1,00 € por litro",1.00,LITRO,https://www.compraonline.alcampo.es/products/p...,2026-10-02T13:21:15
9,leche,Alcampo,95939,PRÉSIDENT Leche semidesnatada de vaca de orige...,1 l,"Precio 0,93 €",0.93,"0,93 € por litro",0.93,LITRO,https://www.compraonline.alcampo.es/products/p...,2026-10-02T13:21:15


## Qué comprobar en la celda anterior

Antes de lanzar las 20 búsquedas, comprueba que:

- aparecen productos;
- `name` contiene el nombre real del artículo;
- `price` contiene el precio total;
- `price_per_unit` aparece cuando Alcampo lo muestra;
- `unit` es `KILO`, `LITRO`, `UNIDAD`, etc.;
- `product_id` se obtiene de la URL del producto;
- las URLs comienzan por `https://www.compraonline.alcampo.es/products/...`.

Si esta prueba funciona, ejecuta las siguientes celdas.  
Si falla, **no seguimos todavía**: revisamos el HTML real que está devolviendo Alcampo y ajustamos selectores.

In [ ]:
# --------------------------------------------------
# SCRAPING DE TODAS LAS BÚSQUEDAS
# --------------------------------------------------
#
# IMPORTANTE:
# Mientras TEST_MODE = True solo ejecutará TEST_TERMS.
# Cuando la prueba de "leche" funcione:
#
# TEST_MODE = False
#
# y vuelve a ejecutar desde la celda de configuración.
# --------------------------------------------------

all_results = []
summary = []

driver = start_alcampo_driver()

try:
    time.sleep(3)
    accept_alcampo_cookies(driver)

    for i, term in enumerate(
        TERMS_TO_SCRAPE,
        start=1
    ):
        print(
            "\n" + "=" * 70
        )
        print(
            f"[{i}/{len(TERMS_TO_SCRAPE)}] "
            f"Buscando: {term}"
        )

        start = time.monotonic()

        try:
            search_alcampo(
                driver,
                term
            )

            loaded_count, load_status = (
                load_all_alcampo_products(
                    driver
                )
            )

            df_term = extract_alcampo_products(
                driver,
                term
            )

            elapsed = round(
                time.monotonic() - start,
                2
            )

            all_results.append(
                df_term
            )

            summary.append({
                "searched_item": term,
                "status": load_status,
                "products_loaded": loaded_count,
                "rows_extracted": len(df_term),
                "seconds": elapsed,
            })

            print(
                f"✓ {len(df_term)} filas extraídas "
                f"en {elapsed} s"
            )

            time.sleep(2)

        except Exception as exc:
            elapsed = round(
                time.monotonic() - start,
                2
            )

            print(
                f"ERROR con '{term}': "
                f"{type(exc).__name__}: {exc}"
            )

            summary.append({
                "searched_item": term,
                "status": "error",
                "products_loaded": None,
                "rows_extracted": 0,
                "seconds": elapsed,
            })

            # Paramos para no encadenar errores.
            break

finally:
    driver.quit()


df_alcampo = (
    pd.concat(
        all_results,
        ignore_index=True
    )
    if all_results
    else pd.DataFrame()
)

df_summary = pd.DataFrame(
    summary
)

display(df_summary)

print(
    "\nFilas raw totales:",
    len(df_alcampo)
)

In [ ]:
# --------------------------------------------------
# REVISIÓN DEL DATASET RAW
# --------------------------------------------------

if not df_alcampo.empty:
    display(
        df_alcampo.head(30)
    )

    print(
        "Filas sin product_id:",
        df_alcampo["product_id"]
        .isna()
        .sum()
    )

    print(
        "IDs distintos:",
        df_alcampo["product_id"]
        .nunique()
    )

    print("\nFilas por búsqueda:")

    display(
        df_alcampo
        .groupby("searched_item")
        .size()
        .rename("filas")
    )

else:
    print(
        "No hay datos para revisar."
    )

In [ ]:
# --------------------------------------------------
# GUARDAR UN ÚNICO CSV RAW
# --------------------------------------------------

if not df_alcampo.empty:
    raw_path = (
        RAW_DIR
        / "alcampo_multiple_searches.csv"
    )

    df_alcampo.to_csv(
        raw_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "CSV raw guardado:",
        raw_path
    )

In [ ]:
# --------------------------------------------------
# DATASET PROCESADO: ELIMINAR DUPLICADOS
# --------------------------------------------------

if not df_alcampo.empty:

    df_alcampo_clean = (
        df_alcampo.copy()
    )

    print(
        "Filas raw:",
        len(df_alcampo_clean)
    )

    # Preferimos deduplicar por ID.
    # Si excepcionalmente faltase algún ID,
    # se conservará de momento para revisarlo.

    with_id = (
        df_alcampo_clean[
            df_alcampo_clean[
                "product_id"
            ].notna()
        ]
        .drop_duplicates(
            subset=["product_id"]
        )
    )

    without_id = (
        df_alcampo_clean[
            df_alcampo_clean[
                "product_id"
            ].isna()
        ]
        .drop_duplicates(
            subset=[
                "name",
                "price",
                "quantity_raw"
            ]
        )
    )

    df_alcampo_clean = (
        pd.concat(
            [with_id, without_id],
            ignore_index=True
        )
        .reset_index(drop=True)
    )

    print(
        "Productos únicos:",
        len(df_alcampo_clean)
    )

    print(
        "IDs únicos:",
        df_alcampo_clean[
            "product_id"
        ].nunique()
    )

else:
    df_alcampo_clean = pd.DataFrame()

In [ ]:
# --------------------------------------------------
# VALIDACIÓN DE DUPLICADOS
# --------------------------------------------------

if not df_alcampo_clean.empty:

    duplicated_ids = (
        df_alcampo_clean[
            df_alcampo_clean[
                "product_id"
            ].notna()
            & df_alcampo_clean.duplicated(
                subset=["product_id"],
                keep=False
            )
        ]
    )

    possible_duplicates = (
        df_alcampo_clean[
            df_alcampo_clean.duplicated(
                subset=["name", "price"],
                keep=False
            )
        ]
        .sort_values(
            ["name", "price"]
        )
    )

    print(
        "Filas con product_id duplicado:",
        len(duplicated_ids)
    )

    print(
        "Filas con mismo nombre y precio:",
        len(possible_duplicates)
    )

    if not possible_duplicates.empty:
        display(
            possible_duplicates[
                [
                    "product_id",
                    "name",
                    "price",
                    "price_per_unit",
                    "unit",
                    "url",
                ]
            ].head(100)
        )

In [ ]:
# --------------------------------------------------
# REVISIÓN DE UNIDADES
# --------------------------------------------------

if not df_alcampo_clean.empty:

    print(
        df_alcampo_clean["unit"]
        .value_counts(
            dropna=False
        )
    )

    unknown_units = (
        df_alcampo_clean[
            df_alcampo_clean[
                "price_per_unit"
            ].isna()
        ]
    )

    print(
        "\nProductos sin precio por unidad detectado:",
        len(unknown_units)
    )

    display(
        unknown_units[
            [
                "name",
                "quantity_raw",
                "price",
                "price_per_unit_raw",
                "unit",
                "url",
            ]
        ].head(50)
    )

In [ ]:
# --------------------------------------------------
# GUARDAR UN ÚNICO CSV PROCESADO
# --------------------------------------------------

if not df_alcampo_clean.empty:

    processed_path = (
        PROCESSED_DIR
        / "alcampo_products.csv"
    )

    df_alcampo_clean.to_csv(
        processed_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "CSV procesado guardado:",
        processed_path
    )

    print(
        "Productos:",
        len(df_alcampo_clean)
    )

## Próximo paso

No conviene adaptar más cosas antes de ver el output real de Alcampo.

Primero ejecuta el notebook con:

```python
TEST_MODE = True
TEST_TERMS = ["leche"]
```

y pásame especialmente:

- `Productos iniciales detectados`
- los resultados de los `Scroll`
- `Enlaces /products/ detectados`
- `Productos extraídos`
- las primeras filas de `df_test`

Con eso podremos confirmar si la extracción funciona o ajustar únicamente los selectores necesarios.